# Nuclear Waste Canister Temperature Prediction — Random Forest Ensemble
**CIVIL-226 - Introduction to Machine Learning for Engineers — EPFL**

**Members:** Nour NAJA | Alessandro CLERICI

## Objective
Predict temperature around nuclear waste canisters at unobserved sensor positions using two Random Forest models trained separately on the Buffer (x < 1.4m) and OPA (x >= 1.4m) zones.

The dataset contains 3 experimental replications per sensor/timestep. The test set corresponds exclusively to Rep 2 (radioactive decay 1488W to 299W), so we train only on Rep 2 data.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

## 2. Load Data

In [ ]:
sensors = pd.read_parquet('data/sensors.parquet')
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')

# Remove duplicate sensors (N206, N213 - identical coordinates, reporting error)
n_before = len(sensors)
sensors  = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)
print(f'Duplicate sensors removed: {n_before - len(sensors)} (N206, N213)')
print(f'Sensors : {sensors.shape}  ->  {sensors.columns.tolist()}')
print(f'Train   : {train.shape}   ->  {train.columns.tolist()}')
print(f'Test    : {test.shape}    ->  {test.columns.tolist()}')

## 3. Data Exploration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train['temperature'].dropna(), bins=50, color='steelblue', edgecolor='white', range=(-200, 200))
axes[0].set_title('Temperature distribution')
axes[0].set_xlabel('Temperature [C]'); axes[0].set_ylabel('Count')

temp_by_time = train.groupby('time')['temperature'].mean()
axes[1].plot(temp_by_time.index / 1e6, temp_by_time.values, color='tomato', linewidth=0.8)
axes[1].set_title('Mean temperature over time')
axes[1].set_xlabel('Time (x10^6 s)'); axes[1].set_ylabel('Mean temperature [C]')
plt.tight_layout(); plt.show()

# Sensor positions
s_train = sensors[sensors['sensor'].isin(set(train['sensor'].unique()))]
s_test  = sensors[sensors['sensor'].isin(set(test['sensor'].unique()))]

plt.figure(figsize=(8, 4))
plt.scatter(s_train['coor_x'], s_train['coor_y'], c='steelblue', s=20, alpha=0.6, label='Train sensors')
plt.scatter(s_test['coor_x'],  s_test['coor_y'],  c='tomato',    s=40, alpha=0.9, marker='^', label='Test sensors')
plt.axvline(1.4, color='gray', linestyle='--', alpha=0.5, label='Buffer/OPA boundary')
plt.xlabel('coor_x [m]'); plt.ylabel('coor_y [m]')
plt.title('Sensor positions (2D)')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Preprocessing

### 4.1 Merge Sensor Positions & Remove NaN

In [ ]:
train_full = train.merge(sensors, on='sensor', how='left')
print(f'Rows before NaN removal: {len(train_full)}')
train_full = train_full.dropna(subset=['temperature']).copy()
print(f'Rows after NaN removal : {len(train_full)}')

### 4.2 Outlier Removal

Two complementary methods:
- **Global MAD** (Median Absolute Deviation): catches physically impossible temperatures
- **Local IQR per timestep**: catches values deviating too much from the distribution at a given moment

In [ ]:
# Global outliers via MAD
temp        = train_full['temperature']
median_temp = temp.median()
mad_temp    = np.median(np.abs(temp - median_temp))
robust_z    = 0.6745 * (temp - median_temp) / (mad_temp + 1e-8)
train_full['is_global_outlier'] = robust_z.abs() > 6
print(f'Global outliers (MAD): {train_full["is_global_outlier"].sum()}')

# Local outliers per timestep via IQR
time_stats = (
    train_full.groupby('time')['temperature']
    .agg(time_median='median',
         time_q1=lambda x: x.quantile(0.25),
         time_q3=lambda x: x.quantile(0.75))
    .reset_index()
)
time_stats['time_iqr'] = time_stats['time_q3'] - time_stats['time_q1']
train_full = train_full.merge(time_stats, on='time', how='left')
train_full['is_local_outlier'] = (
    (train_full['temperature'] < train_full['time_median'] - 4.0 * train_full['time_iqr']) |
    (train_full['temperature'] > train_full['time_median'] + 4.0 * train_full['time_iqr'])
)
print(f'Local outliers (IQR): {train_full["is_local_outlier"].sum()}')

before_rows = len(train_full)
train_full  = train_full[
    (~train_full['is_global_outlier']) & (~train_full['is_local_outlier'])
].copy()
print(f'Rows removed: {before_rows - len(train_full)} | Remaining: {len(train_full)}')
train_full = train_full.drop(columns=['is_global_outlier', 'is_local_outlier',
                                       'time_median', 'time_q1', 'time_q3', 'time_iqr'])

### 4.3 Sensor Drift Detection & Correction

Some sensors show a systematic linear drift in their residuals over time. We detect and correct this before training.

In [ ]:
time_median = (
    train_full.groupby('time')['temperature']
    .median().rename('global_time_median').reset_index()
)
drift_df = train_full.merge(time_median, on='time', how='left').copy()
drift_df['temp_residual'] = drift_df['temperature'] - drift_df['global_time_median']

t_min = drift_df['time'].min()
t_max = drift_df['time'].max()
drift_df['time_norm_for_drift'] = (drift_df['time'] - t_min) / (t_max - t_min + 1e-8)

drift_records = []
for sensor_id, g in drift_df.groupby('sensor'):
    g = g.sort_values('time_norm_for_drift')
    if len(g) < 20:
        continue
    x = g['time_norm_for_drift'].values
    y = g['temp_residual'].values
    slope = np.polyfit(x, y, 1)[0] if np.std(y) > 1e-8 else 0.0
    corr  = np.corrcoef(x, y)[0, 1] if np.std(y) > 1e-8 else 0.0
    drift_records.append({'sensor': sensor_id, 'drift_slope': slope, 'drift_corr': corr})

sensor_drift    = pd.DataFrame(drift_records)
slope_abs       = sensor_drift['drift_slope'].abs()
slope_threshold = slope_abs.median() + 4 * (np.median(np.abs(slope_abs - slope_abs.median())) + 1e-8)
sensor_drift['is_drift_sensor'] = (
    (sensor_drift['drift_slope'].abs() > slope_threshold) &
    (sensor_drift['drift_corr'].abs() > 0.5)
)
drift_sensors = sensor_drift.loc[sensor_drift['is_drift_sensor'], 'sensor'].unique()
print(f'Drift sensors detected: {len(drift_sensors)}')

t_min = train_full['time'].min()
t_max = train_full['time'].max()
train_full['time_norm_for_drift'] = (train_full['time'] - t_min) / (t_max - t_min + 1e-8)
drift_map = sensor_drift.set_index('sensor')['drift_slope'].to_dict()
train_full['drift_correction'] = 0.0
for s in drift_sensors:
    mask = train_full['sensor'] == s
    train_full.loc[mask, 'drift_correction'] = (
        drift_map[s] * (train_full.loc[mask, 'time_norm_for_drift'] - 0.5)
    )
train_full['temperature'] = train_full['temperature'] - train_full['drift_correction']
train_full = train_full.drop(columns=['time_norm_for_drift', 'drift_correction'])
print('Drift correction applied.')

### 4.4 Filter Replication 2

Each (sensor, timestep) has exactly 3 rows, one per experimental replication:
- **Rep 0**: 500W to 0W (drops around year 100)
- **Rep 1**: 1400W to 0W (drops around year 110)
- **Rep 2**: radioactive decay 1488W to 299W over 250 years

The test set matches **Rep 2 exclusively**, so we train only on Rep 2 to avoid noise from physically incompatible power scenarios.

In [ ]:
train_full['rep'] = (
    train_full.groupby(['sensor', 'time'])['power']
    .rank(method='first').astype(int) - 1
)
for rep in [0, 1, 2]:
    sub = train_full[train_full['rep'] == rep]
    print(f'Rep {rep}: {sub.sensor.nunique()} sensors, {len(sub):,} rows')

train_full  = train_full[train_full['rep'] == 2].drop(columns=['rep']).copy()
train_clean = train_full.copy()
print(f'\nAfter filtering Rep 2: {len(train_full):,} rows, {train_full.sensor.nunique()} sensors')

### 4.5 Feature Engineering

Physics-motivated features:
- **dist_canister**: temperature decays with distance from the heat source
- **is_opa**: zone indicator (OPA weighted more in Kaggle score)
- **time_log**: captures logarithmic thermal diffusion dynamics
- **power_x_time, dist_x_time**: physical interactions between power, distance and time
- **inv_dist_canister**: radial gradient near the canister
- **coor_x_squared**: spatial non-linearity in x

In [ ]:
t_max_ref = train_full['time'].max()

def add_features(df, sensors_df):
    """
    Add physics-motivated engineered features.
    Merges sensor coordinates if not already present (e.g. test set).
    """
    merged = df.copy()
    if 'coor_x' not in merged.columns:
        merged = merged.merge(sensors_df, on='sensor', how='left')

    t_max = t_max_ref

    # Spatial features
    merged['dist_center']       = np.sqrt(merged['coor_x']**2 + merged['coor_y']**2)
    merged['dist_canister']     = np.sqrt((merged['coor_x'] - 0.7)**2 + (merged['coor_y'] - 1.2)**2)
    merged['is_opa']            = (merged['coor_x'] > 1.4).astype(float)
    merged['inv_dist_canister'] = 1 / (merged['dist_canister'] + 0.1)
    merged['coor_x_squared']    = merged['coor_x'] ** 2

    # Temporal features
    merged['time_norm'] = merged['time'] / t_max
    merged['time_log']  = np.log1p(merged['time'])

    # Physical interaction features
    merged['power_x_time'] = merged['power'] * merged['time_norm']
    merged['dist_x_time']  = merged['dist_canister'] * merged['time_norm']
    merged['x_x_time']     = merged['coor_x'] * merged['time_norm']
    merged['dist_x_log']   = merged['dist_canister'] * merged['time_log']
    merged['time_x_dist']  = merged['time_norm'] * merged['dist_canister']

    return merged

train_feat = add_features(train_full, sensors)
print('Features:', train_feat.columns.tolist())
display(train_feat.head(3))

### 4.6 Feature List & Buffer/OPA Zone Split

The Buffer (x < 1.4m) and OPA (x >= 1.4m) zones have fundamentally different thermal properties. Training two separate RF models allows each to specialize on its zone dynamics.

In [ ]:
TARGET   = 'temperature'
FEATURES = [
    'coor_x', 'coor_y', 'time_norm', 'time_log', 'power',
    'dist_center', 'dist_canister', 'is_opa',
    'power_x_time', 'dist_x_time', 'x_x_time', 'dist_x_log',
    'inv_dist_canister', 'coor_x_squared', 'time_x_dist'
]

BUFFER_THRESHOLD = 1.4

mask_buf = train_feat['coor_x'] <  BUFFER_THRESHOLD
mask_opa = train_feat['coor_x'] >= BUFFER_THRESHOLD

X_buf = train_feat.loc[mask_buf, FEATURES].values
y_buf = train_feat.loc[mask_buf, TARGET].values
X_opa = train_feat.loc[mask_opa, FEATURES].values
y_opa = train_feat.loc[mask_opa, TARGET].values

print(f'Buffer: {len(X_buf):,} rows | OPA: {len(X_opa):,} rows')

### 4.7 Train/Validation Split & Normalisation

In [ ]:
X_train_buf, X_val_buf, y_train_buf, y_val_buf = train_test_split(X_buf, y_buf, test_size=0.2, random_state=42)
X_train_opa, X_val_opa, y_train_opa, y_val_opa = train_test_split(X_opa, y_opa, test_size=0.2, random_state=42)

# Fit scaler ONLY on training data, separate scaler per zone
scaler_buf = StandardScaler()
X_train_buf_s = scaler_buf.fit_transform(X_train_buf)
X_val_buf_s   = scaler_buf.transform(X_val_buf)

scaler_opa = StandardScaler()
X_train_opa_s = scaler_opa.fit_transform(X_train_opa)
X_val_opa_s   = scaler_opa.transform(X_val_opa)

print(f'Buffer: Train {X_train_buf_s.shape} | Val {X_val_buf_s.shape}')
print(f'OPA   : Train {X_train_opa_s.shape} | Val {X_val_opa_s.shape}')

## 5. Model — Random Forest Ensemble

Two independent Random Forest models, one per zone:

$$\hat{y} = \frac{1}{T}\sum_{t=1}^{T} h_t(\mathbf{x})$$

Splitting by zone allows each model to specialize on its zone's thermal dynamics, reducing error at the Buffer/OPA interface and in the OPA zone where a single model struggled.

In [ ]:
rf_buf = RandomForestRegressor(
    n_estimators=100, max_depth=15, min_samples_leaf=10, n_jobs=-1, random_state=42
)
rf_buf.fit(X_train_buf_s, y_train_buf)
y_pred_val_buf = rf_buf.predict(X_val_buf_s)
rmse_buf = np.sqrt(mean_squared_error(y_val_buf, y_pred_val_buf))
print(f'Buffer RF: RMSE validation = {rmse_buf:.4f} C')

rf_opa = RandomForestRegressor(
    n_estimators=100, max_depth=15, min_samples_leaf=10, n_jobs=-1, random_state=42
)
rf_opa.fit(X_train_opa_s, y_train_opa)
y_pred_val_opa = rf_opa.predict(X_val_opa_s)
rmse_opa = np.sqrt(mean_squared_error(y_val_opa, y_pred_val_opa))
print(f'OPA    RF: RMSE validation = {rmse_opa:.4f} C')

y_val_all   = np.concatenate([y_val_buf, y_val_opa])
y_pred_all  = np.concatenate([y_pred_val_buf, y_pred_val_opa])
rmse_global = np.sqrt(mean_squared_error(y_val_all, y_pred_all))
print(f'Global   : RMSE validation = {rmse_global:.4f} C')

## 6. Evaluation

In [ ]:
# Feature importance — both zones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, rf, zone in zip(axes, [rf_buf, rf_opa], ['Buffer', 'OPA']):
    imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
    imp.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Feature Importance: {zone}')
    ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

In [ ]:
# True vs predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(y_val_all, y_pred_all, s=3, alpha=0.2)
mn, mx = min(y_val_all.min(), y_pred_all.min()), max(y_val_all.max(), y_pred_all.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('True temperature [C]')
plt.ylabel('Predicted temperature [C]')
plt.title(f'RF Ensemble: True vs Predicted (RMSE={rmse_global:.2f} C)')
plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
# Validation over time
_, idx_val_buf = train_test_split(np.arange(len(X_buf)), test_size=0.2, random_state=42)
_, idx_val_opa = train_test_split(np.arange(len(X_opa)), test_size=0.2, random_state=42)

val_feat_buf = train_feat[mask_buf].iloc[idx_val_buf].copy().reset_index(drop=True)
val_feat_opa = train_feat[mask_opa].iloc[idx_val_opa].copy().reset_index(drop=True)
val_feat_buf['temp_pred'] = y_pred_val_buf
val_feat_opa['temp_pred'] = y_pred_val_opa
val_feat_all = pd.concat([val_feat_buf, val_feat_opa], ignore_index=True)

temp_val_true = val_feat_all.groupby('time')['temperature'].mean()
temp_val_pred = val_feat_all.groupby('time')['temp_pred'].mean()

plt.figure(figsize=(10, 4))
plt.plot(temp_val_true.index / 1e6, temp_val_true.values,
         label='Validation (true)', color='black', alpha=0.7, linewidth=0.8)
plt.plot(temp_val_pred.index / 1e6, temp_val_pred.values,
         label='Validation (predicted)', color='tomato', alpha=0.7, linewidth=0.8)
plt.title('Validation: true vs predicted over time')
plt.xlabel('Time (x10^6 s)'); plt.ylabel('Mean temperature [C]')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Spatial error map
val_feat_all['abs_error'] = np.abs(val_feat_all['temperature'] - val_feat_all['temp_pred'])
val_feat_all['sq_error']  = (val_feat_all['temperature'] - val_feat_all['temp_pred'])**2

sensor_metrics = (
    val_feat_all
    .groupby('sensor')
    .agg(
        mae   =('abs_error', 'mean'),
        rmse  =('sq_error',  lambda x: np.sqrt(np.mean(x))),
        coor_x=('coor_x', 'first'),
        coor_y=('coor_y', 'first'),
    )
    .sort_values('rmse', ascending=False)
)
print('Worst sensors by RMSE:')
print(sensor_metrics.head(10))

plt.figure(figsize=(8, 5))
sc = plt.scatter(
    sensor_metrics['coor_x'], sensor_metrics['coor_y'],
    c=sensor_metrics['rmse'], cmap='hot_r', s=80
)
plt.colorbar(sc, label='RMSE [C]')
plt.axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA boundary')
plt.xlabel('coor_x'); plt.ylabel('coor_y')
plt.title('Validation RMSE by sensor position')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 7. Final Predictions & Submission

In [ ]:
test_feat = add_features(test, sensors)

mask_buf_test = test_feat['coor_x'] <  BUFFER_THRESHOLD
mask_opa_test = test_feat['coor_x'] >= BUFFER_THRESHOLD

X_test_buf_s = scaler_buf.transform(test_feat.loc[mask_buf_test, FEATURES].values)
X_test_opa_s = scaler_opa.transform(test_feat.loc[mask_opa_test, FEATURES].values)

y_pred = np.zeros(len(test_feat))
y_pred[mask_buf_test.values] = rf_buf.predict(X_test_buf_s)
y_pred[mask_opa_test.values] = rf_opa.predict(X_test_opa_s)

submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})
assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'Buffer RMSE: {rmse_buf:.4f} | OPA RMSE: {rmse_opa:.4f} | Global RMSE: {rmse_global:.4f}')
print(f'submission.csv saved: {len(submission)} rows')
display(submission.head())

In [ ]:
# Sanity check: train vs test predictions over time
test_feat['temp_pred'] = y_pred
temp_train = train_clean.groupby('time')['temperature'].mean()
temp_pred  = test_feat.groupby('time')['temp_pred'].mean()

plt.figure(figsize=(10, 4))
plt.plot(temp_train.index / 1e6, temp_train.values,
         label='Train Rep2 (true)', color='tomato', alpha=0.7, linewidth=0.8)
plt.plot(temp_pred.index / 1e6, temp_pred.values,
         label='Test (predicted)', color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Sanity check: Train vs Test predictions over time')
plt.xlabel('Time (x10^6 s)'); plt.ylabel('Mean temperature [C]')
plt.legend(); plt.tight_layout(); plt.show()

## 8. Spatial Temperature Map: True vs Predicted

In [ ]:
train_sensor_temp = (
    train_clean
    .groupby('sensor')
    .agg(coor_x=('coor_x', 'first'), coor_y=('coor_y', 'first'), mean_temp=('temperature', 'mean'))
    .reset_index()
)
test_sensor_temp = (
    test_feat
    .groupby('sensor')
    .agg(coor_x=('coor_x', 'first'), coor_y=('coor_y', 'first'), mean_temp=('temp_pred', 'mean'))
    .reset_index()
)

vmin = min(train_sensor_temp['mean_temp'].min(), test_sensor_temp['mean_temp'].min())
vmax = max(train_sensor_temp['mean_temp'].max(), test_sensor_temp['mean_temp'].max())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(
    train_sensor_temp['coor_x'], train_sensor_temp['coor_y'],
    c=train_sensor_temp['mean_temp'], cmap='hot_r', vmin=vmin, vmax=vmax,
    s=80, edgecolors='gray', linewidths=0.3
)
axes[0].axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA x=1.4')
axes[0].set_title('True temperatures: Train Rep2 sensors', fontsize=13)
axes[0].set_xlabel('coor_x [m]'); axes[0].set_ylabel('coor_y [m]')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
plt.colorbar(sc1, ax=axes[0], label='Mean temperature [C]')

sc2 = axes[1].scatter(
    test_sensor_temp['coor_x'], test_sensor_temp['coor_y'],
    c=test_sensor_temp['mean_temp'], cmap='hot_r', vmin=vmin, vmax=vmax,
    s=80, edgecolors='gray', linewidths=0.3
)
axes[1].axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA x=1.4')
axes[1].set_title('Predicted temperatures: Test sensors (RF Ensemble)', fontsize=13)
axes[1].set_xlabel('coor_x [m]'); axes[1].set_ylabel('coor_y [m]')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label='Mean temperature [C]')

plt.suptitle('Spatial temperature distribution: true vs predicted\nExpected: radial gradient, hot near canister (x=0), cool far away',
             fontsize=14, y=1.02)
plt.tight_layout(); plt.show()